In [1]:
# ===== P6 CELL 1: Connect Google Drive =====
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ===== P6 CELL 2: Load settings, scores and (for the first time) the labels =====
import os, json, hashlib, datetime, math
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

BASE = os.environ.get("BETH_BASE", "/content/drive/MyDrive/BETH_Triage")
FEAT_DIR = os.path.join(BASE, "02_data_processed", "features")
SPLIT_DIR = os.path.join(BASE, "02_data_processed", "splits")
SCORE_DIR = os.path.join(BASE, "04_scores")
RES_DIR = os.path.join(BASE, "05_results")
OUT_DIR = os.path.join(BASE, "08_triage_output")

with open(os.path.join(BASE, "config.json")) as fh:
    CONFIG = json.load(fh)
proto = CONFIG["protocol"]
with open(os.path.join(BASE, proto["file"]), "rb") as fh:
    assert hashlib.sha256(fh.read()).hexdigest() == proto["sha256"], "Protocol file changed - STOP"
PLAN = proto["plan"]
ALPHAS, A0 = PLAN["alphas"], PLAN["primary_alpha"]
print("Protocol", proto["version"], "verified unchanged")

meta_val = pd.read_pickle(os.path.join(FEAT_DIR, "meta_val.pkl"))
meta_test = pd.read_pickle(os.path.join(FEAT_DIR, "meta_test.pkl"))
with open(os.path.join(SPLIT_DIR, "calibration_folds.json")) as fh:
    FOLDS = json.load(fh)["folds"]

CONFIGS = sorted(f[:-4] for f in os.listdir(SCORE_DIR) if f.endswith(".npz"))
SCORES = {c: dict(np.load(os.path.join(SCORE_DIR, c + ".npz"))) for c in CONFIGS}
assert len(CONFIGS) == 27, f"expected 27 score files, found {len(CONFIGS)}"

y_evil = meta_test["evil"].values.astype(int)
y_sus = meta_test["sus"].values.astype(int)
print(f"{len(CONFIGS)} configurations loaded. Test: {y_evil.sum():,} attack rows, "
      f"{(y_evil == 0).sum():,} normal rows ({((y_evil == 0) & (y_sus == 0)).sum():,} clean normal).")

Protocol v1 verified unchanged
27 configurations loaded. Test: 158,432 attack rows, 30,535 normal rows (17,508 clean normal).


In [3]:
# ===== P6 CELL 3: Core functions - conformal p-values, process scores, metrics =====
def conformal_p(cal_scores, new_scores):
    """p = (1 + number of calibration scores >= new score) / (n + 1)  (protocol section 4)."""
    cal = np.sort(np.asarray(cal_scores))
    n = len(cal)
    n_ge = n - np.searchsorted(cal, np.asarray(new_scores), side="left")
    return (1 + n_ge) / (n + 1)

def process_table(meta, scores):
    """One row per process: score = maximum event score; labels = any event labelled."""
    df = pd.DataFrame({"key": meta["process_key"].values, "s": scores})
    if "evil" in meta:
        df["evil"], df["sus"] = meta["evil"].values, meta["sus"].values
    g = df.groupby("key", sort=True)
    out = pd.DataFrame({"s": g["s"].max()})
    if "evil" in df:
        out["evil"], out["sus"] = g["evil"].max(), g["sus"].max()
    return out

def random_order(scores, seed=42):
    """Highest score first; ties put in a seeded random order (protocol: tie_breaking)."""
    tie = np.random.default_rng(seed).random(len(scores))
    return np.lexsort((tie, -np.asarray(scores)))

def reading_cost(scores, y, level):
    """Normal records ranked above the point where `level` of positives have been reached."""
    order = random_order(scores)
    ys = y[order]
    need = math.ceil(level * ys.sum())
    pos_idx = np.where(ys == 1)[0][need - 1]
    return int((ys[:pos_idx] == 0).sum())

def precision_formula(pi, recall, frr):
    den = pi * recall + (1 - pi) * frr
    return float(pi * recall / den) if den > 0 else float("nan")

def event_metrics(cfg, alpha):
    s_val, s_test = SCORES[cfg]["val"], SCORES[cfg]["test"]
    p = conformal_p(s_val, s_test)
    kept = p <= alpha
    normal_all, normal_clean = y_evil == 0, (y_evil == 0) & (y_sus == 0)
    att = meta_test.loc[y_evil == 1, ["process_key"]].assign(kept=kept[y_evil == 1])
    row = {"config": cfg, "level": "event", "alpha": alpha,
           "FRR_normal_all": kept[normal_all].mean(), "FRR_normal_clean": kept[normal_clean].mean(),
           "recall_evil": kept[y_evil == 1].mean(),
           "macro_recall_evil": att.groupby("process_key")["kept"].mean().mean(),
           "recall_sus": kept[y_sus == 1].mean(), "FRR_sus0": kept[y_sus == 0].mean(),
           "queue_size": int(kept.sum())}
    for pi in PLAN["prevalences_for_precision_formula"]:
        row[f"precision_at_{int(pi * 100)}pct"] = precision_formula(pi, row["recall_evil"], row["FRR_normal_all"])
    return row

def process_metrics(cfg, alpha):
    pv = process_table(meta_val, SCORES[cfg]["val"])
    pt = process_table(meta_test, SCORES[cfg]["test"])
    reachable = alpha >= 1 / (len(pv) + 1)
    kept = conformal_p(pv["s"].values, pt["s"].values) <= alpha
    ev, su = pt["evil"].values == 1, pt["sus"].values == 1
    return {"config": cfg, "level": "process", "alpha": alpha, "reachable": reachable,
            "FRR_normal_all": kept[~ev].mean(), "FRR_normal_clean": kept[~ev & ~su].mean(),
            "recall_evil": kept[ev].mean(), "recall_sus": kept[su].mean(), "FRR_sus0": kept[~su].mean(),
            "queue_size": int(kept.sum()), "attack_processes": int(ev.sum()), "normal_processes": int((~ev).sum())}

# quick self-test of the p-value formula on a tiny example
assert np.allclose(conformal_p([1, 2, 3, 4], [4.5, 2, 0]), [1 / 5, 4 / 5, 5 / 5])
print("Functions ready; p-value self-test passed.")

Functions ready; p-value self-test passed.


In [4]:
# ===== P6 CELL 4: Hypothesis H1 - does the false-retention promise hold on unseen hosts? =====
rows = []
for cfg in CONFIGS:
    s_val = SCORES[cfg]["val"]
    for fold in FOLDS:
        cal_mask = meta_val["hostName"].isin(fold["calibration_hosts"]).values
        p_held = conformal_p(s_val[cal_mask], s_val[~cal_mask])
        pv_cal = process_table(meta_val[cal_mask], s_val[cal_mask])
        pv_held = process_table(meta_val[~cal_mask], s_val[~cal_mask])
        p_proc = conformal_p(pv_cal["s"].values, pv_held["s"].values)
        for a in ALPHAS:
            rows.append({"config": cfg, "held_out_host": fold["held_out_host"], "alpha": a,
                         "FRR_event": (p_held <= a).mean(),
                         "FRR_process": (p_proc <= a).mean() if a >= 1 / (len(pv_cal) + 1) else np.nan})
h1 = pd.DataFrame(rows)
h1.to_csv(os.path.join(RES_DIR, "P6_H1_leave_one_host_out.csv"), index=False)

prim = h1[h1["alpha"] == A0]
summary = prim.groupby("config").agg(worst_FRR_event=("FRR_event", "max"),
                                     mean_FRR_event=("FRR_event", "mean"),
                                     worst_FRR_process=("FRR_process", "max"))
summary["H1_event (worst <= 2a)"] = np.where(summary["worst_FRR_event"] <= 2 * A0, "supported", "NOT supported")
summary.to_csv(os.path.join(RES_DIR, "P6_H1_summary.csv"))
pd.set_option("display.width", 220); pd.set_option("display.max_rows", 100)
print(f"H1 at alpha = {A0} (target: worst held-out host FRR <= {2 * A0}):\n")
print(summary.round(4).to_string())

H1 at alpha = 0.01 (target: worst held-out host FRR <= 0.02):

                             worst_FRR_event  mean_FRR_event  worst_FRR_process H1_event (worst <= 2a)
config                                                                                                
autoencoder__F_main__s42              0.0504          0.0349             0.0270          NOT supported
autoencoder__F_main__s43              0.0502          0.0359             0.0270          NOT supported
autoencoder__F_main__s44              0.0562          0.0406             0.0270          NOT supported
autoencoder__F_main__s45              0.0489          0.0351             0.0476          NOT supported
autoencoder__F_main__s46              0.0496          0.0359             0.0270          NOT supported
autoencoder__F_no_user__s42           0.0572          0.0398             0.0270          NOT supported
autoencoder__F_paper__s42             0.0394          0.0276             0.0000          NOT supported
autoencode

In [5]:
# ===== P6 CELL 5: Primary test metrics (unmodified test data) for every configuration =====
ev_rows, pr_rows, rank_rows = [], [], []
for cfg in CONFIGS:
    for a in ALPHAS:
        ev_rows.append(event_metrics(cfg, a))
        pr_rows.append(process_metrics(cfg, a))
    s_test = SCORES[cfg]["test"]
    pt = process_table(meta_test, s_test)
    rr = {"config": cfg,
          "AUROC_event_evil": roc_auc_score(y_evil, s_test),
          "AUROC_event_sus": roc_auc_score(y_sus, s_test),
          "AUROC_process_evil": roc_auc_score(pt["evil"], pt["s"]),
          "AUROC_process_sus": roc_auc_score(pt["sus"], pt["s"])}
    for lvl in PLAN["recall_levels_for_reading_cost"]:
        rr[f"normal_read_before_{int(lvl * 100)}pct_attack"] = reading_cost(s_test, y_evil, lvl)
    rank_rows.append(rr)

ev_df, pr_df, rank_df = pd.DataFrame(ev_rows), pd.DataFrame(pr_rows), pd.DataFrame(rank_rows)
ev_df.to_csv(os.path.join(RES_DIR, "P6_test_metrics_event.csv"), index=False)
pr_df.to_csv(os.path.join(RES_DIR, "P6_test_metrics_process.csv"), index=False)
rank_df.to_csv(os.path.join(RES_DIR, "P6_test_ranking_metrics.csv"), index=False)

HEADLINE = ["random__none__s42", "rule_userId__none__s0", "rarity__none__s0",
            "ocsvm__F_main__s42", "iforest__F_main__s42", "autoencoder__F_main__s42"]
e0 = ev_df[(ev_df["alpha"] == A0) & ev_df["config"].isin(HEADLINE)].set_index("config")
p0 = pr_df[(pr_df["alpha"] == A0) & pr_df["config"].isin(HEADLINE)].set_index("config")
head = pd.DataFrame({
    "FRR_event (all)": e0["FRR_normal_all"], "FRR_event (clean)": e0["FRR_normal_clean"],
    "recall_event": e0["recall_evil"], "macro_recall": e0["macro_recall_evil"],
    "recall_process": p0["recall_evil"], "FRR_process": p0["FRR_normal_all"],
    "precision@1%": e0["precision_at_1pct"], "precision@5%": e0["precision_at_5pct"],
}).join(rank_df.set_index("config")[["AUROC_event_evil", "AUROC_process_evil",
                                     "normal_read_before_50pct_attack", "normal_read_before_90pct_attack"]]).loc[HEADLINE]
head.to_csv(os.path.join(RES_DIR, "P6_headline_alpha001.csv"))
print(f"Headline results at alpha = {A0} (seed 42, F_main for models). Point estimates only -")
print("confidence intervals and significance tests come in Phase 7.\n")
print(head.round(4).to_string())

Headline results at alpha = 0.01 (seed 42, F_main for models). Point estimates only -
confidence intervals and significance tests come in Phase 7.

                          FRR_event (all)  FRR_event (clean)  recall_event  macro_recall  recall_process  FRR_process  precision@1%  precision@5%  AUROC_event_evil  AUROC_process_evil  normal_read_before_50pct_attack  normal_read_before_90pct_attack
config                                                                                                                                                                                                                                                   
random__none__s42                  0.0105             0.0115        0.0098        0.0082             0.0          0.0        0.0094        0.0470            0.4993              0.4427                            15246                            27544
rule_userId__none__s0              0.1296             0.0000        0.9928        0.7190             0

In [6]:
# ===== P6 CELL 6: Export the forensic triage queues (no labels) and the shared-schema files =====
from sklearn.preprocessing import StandardScaler
with open(os.path.join(FEAT_DIR, "feature_names.json")) as fh:
    FNAMES = np.array(json.load(fh)["F_main"])
Xtr = np.load(os.path.join(FEAT_DIR, "F_main_train.npy"))
mu, sd = Xtr.mean(axis=0), Xtr.std(axis=0)
sd[sd == 0] = 1.0
del Xtr
X_test = np.load(os.path.join(FEAT_DIR, "F_main_test.npy"))

def explain(rows_idx, k=3):
    """Top-k features that deviate most from the training average (|z| score)."""
    z = np.abs((X_test[rows_idx] - mu) / sd)
    top = np.argsort(-z, axis=1)[:, :k]
    return ["; ".join(FNAMES[t]) for t in top]

for model in ["iforest", "autoencoder"]:
    cfg = f"{model}__F_main__s42"
    s_val, s_test = SCORES[cfg]["val"], SCORES[cfg]["test"]
    p = conformal_p(s_val, s_test)
    keep = np.where(p <= A0)[0]
    q = meta_test.iloc[keep][["row_id", "timestamp", "hostName", "processId", "processName",
                              "eventName", "userId", "process_key"]].copy()
    q["Anomaly_Score"] = s_test[keep]
    q["p_value"] = p[keep]
    q["Investigation_Priority_Score"] = 1 - p[keep]
    q["Triage_Result"] = f"Retained (p <= {A0})"
    q["Explanation_top_deviating_features"] = explain(keep)
    q = q.sort_values(["Anomaly_Score", "timestamp"], ascending=[False, True]).reset_index(drop=True)
    q.insert(0, "Rank", np.arange(1, len(q) + 1))
    q.insert(1, "Evidence_Reference", "labelled_testing_data.csv#row_id=" + q["row_id"].astype(str))
    q.to_csv(os.path.join(OUT_DIR, f"Forensic_Triage_Queue_{model}.csv"), index=False)

    shared = pd.DataFrame({
        "Finding_ID": [f"M3-{model[:2].upper()}-{i:06d}" for i in q["Rank"]],
        "Source_Component": "Member 3 - Unsupervised Forensic Evidence Triage",
        "Timestamp": q["timestamp"], "Entity_Resource_Context":
            "host=" + q["hostName"] + "; process=" + q["processName"] + " (pid " + q["processId"].astype(str)
            + "); event=" + q["eventName"] + "; userId=" + q["userId"].astype(str),
        "Prediction_Result": "Prioritized anomalous evidence candidate (not a confirmed attack)",
        "Confidence_Priority": q["Investigation_Priority_Score"].round(6),
        "Evidence_Reference": q["Evidence_Reference"]})
    shared.to_csv(os.path.join(OUT_DIR, f"Shared_Security_Findings_Member3_{model}.csv"), index=False)
    print(f"{model}: {len(q):,} of {len(meta_test):,} test records retained "
          f"({100 * (1 - len(q) / len(meta_test)):.1f}% evidence volume reduction on the original file)")
    print("   top 3 rows of the queue:")
    print(q.head(3)[["Rank", "processName", "eventName", "Investigation_Priority_Score",
                     "Explanation_top_deviating_features"]].to_string(index=False))
print("\nQueues contain no labels: they are what an investigator would receive.")

iforest: 762 of 188,967 test records retained (99.6% evidence volume reduction on the original file)
   top 3 rows of the queue:
 Rank processName   eventName  Investigation_Priority_Score                Explanation_top_deviating_features
    1     systemd cap_capable                      0.999963 event=cap_capable; is_os_process; process=systemd
    2     systemd      socket                      0.999963      event=socket; is_os_process; process=systemd
    3     systemd      socket                      0.999963      event=socket; is_os_process; process=systemd
autoencoder: 171,052 of 188,967 test records retained (9.5% evidence volume reduction on the original file)
   top 3 rows of the queue:
 Rank     processName eventName  Investigation_Priority_Score                     Explanation_top_deviating_features
    1 systemd-user-ru    umount                      0.999968 event=umount; process=systemd-user-ru; process_logfreq
    2          docker     clone                      0.999952